# Wojtek RL guide (Colab)

Step by step: robot → environment → training → export → your policy vs the one on the robot. Every step ends with a MuJoCo view of what just happened.

Runtime: **GPU** (Runtime → Change runtime type). Optional Secrets: `HF_ORGANIZATION`, `HF_TOKEN` (step 7 needs them for published keepers).
Run the cells in order.

## Step 0 — Install

Clones the repo and installs `training/` (JAX + MuJoCo MJX + MJWarp + Brax). If the next cell fails to import, Runtime → Restart session, then run from the top.

In [ ]:
import os, subprocess, sys
from pathlib import Path

if sys.platform == "linux":
    os.environ.setdefault("MUJOCO_GL", "egl")   # headless rendering; before `import mujoco`

REPO_BRANCH = "main"
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "training" / "run.sh").exists()), None)
if REPO_ROOT is None:
    REPO_ROOT = Path.cwd() / "w01-tek"
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "-q", "-b", REPO_BRANCH, "https://github.com/machinekind/w01-tek.git", str(REPO_ROOT)], check=True)
TRAINING = REPO_ROOT / "training"

try:
    from google.colab import userdata  # type: ignore
    for key in ("HF_ORGANIZATION", "HF_TOKEN"):
        try:
            os.environ.setdefault(key, userdata.get(key))
        except Exception:
            pass
except ImportError:
    pass

try:
    import wojtek_rl  # noqa: F401
except ImportError:
    import tomllib
    lock = tomllib.loads((TRAINING / "uv.lock").read_text())
    mujoco_pin = next(p["version"] for p in lock["package"] if p["name"] == "mujoco")   # keep mujoco at the locked version
    res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(TRAINING), f"mujoco=={mujoco_pin}"],
                         capture_output=True, text=True)
    if res.returncode:
        print(res.stdout[-1500:], res.stderr[-3000:])
        raise SystemExit("pip install failed (see above)")
    print("installed; if the next cell fails to import, restart the session and rerun from the top")
print(REPO_ROOT, "| python", sys.version.split()[0])

In [ ]:
import json, re, shutil, time
import matplotlib.pyplot as plt
import mediapy as media
import mujoco
import numpy as np
from IPython.display import Video

if shutil.which("ffmpeg") is None:          # mediapy needs an ffmpeg binary
    import imageio_ffmpeg
    media.set_ffmpeg(imageio_ffmpeg.get_ffmpeg_exe())

if str(TRAINING) not in sys.path:
    sys.path.insert(0, str(TRAINING))
from wojtek_rl import paths

RUN_NAME = "guide_stiff_v1_s0"
PRESET = "locomotion_stiff_v1"      # frozen keeper recipe: IMU-blind, phase-free, exportable
TRAIN_STEPS = 30_000_000            # pipeline check; the keeper trained 2.0B
NUM_ENVS = 4096                     # must divide 8192
SEED = 0
KEEPER_NAME = "wojtek-quiet-locomotion"   # the pin on the robot

RUN_DIR = TRAINING / "runs" / RUN_NAME
EXPORT_DIR = RUN_DIR / "deploy"
OUT = TRAINING / "videos" / "guide"
OUT.mkdir(parents=True, exist_ok=True)

def show(frames, fps=25):
    media.show_video(np.asarray(frames), fps=fps, codec="h264")

def run_module(module, *args, cpu=False, tail=800):
    env = dict(os.environ, JAX_PLATFORMS="cpu") if cpu else dict(os.environ)
    res = subprocess.run([sys.executable, "-m", module, *args], cwd=TRAINING, env=env, capture_output=True, text=True)
    print((res.stdout or res.stderr)[-tail:])
    return res.returncode

import jax
print("jax", jax.__version__, [d.platform for d in jax.devices()], "| HF org", "set" if os.environ.get("HF_ORGANIZATION") else "unset")

## Step 1 — The robot

12 PD position actuators (abduction, hip, knee × 4 legs), four-bar linkage legs, 14 kg. Physics 250 Hz, policy 50 Hz.
Model: `ros/src/wojtek_description/mujoco/scene_mjx.xml`, generated by `./training/run.sh build`. Never edit it by hand.

View: the home pose, then one hip target stepped by +0.2 rad. The servo `tau = kp (q_target - q) - kd q̇` does the rest; `kp`, `kd` and the torque cap are part of every policy.

In [ ]:
model = mujoco.MjModel.from_xml_path(str(paths.SCENE_XML))
data = mujoco.MjData(model)
ACTUATORS = [model.actuator(i).name for i in range(model.nu)]
print(f"nu={model.nu} mass={sum(model.body_mass):.1f} kg dt={model.opt.timestep} | XML servo kp={model.actuator_gainprm[0, 0]:g} kd={-model.actuator_biasprm[0, 2]:g} cap ±{model.actuator_forcerange[0, 1]:g} N·m")
print(ACTUATORS)

renderer = mujoco.Renderer(model, height=360, width=480)
def frame(m, d, cam="track"):
    renderer.update_scene(d, camera=cam)
    return renderer.render().copy()

mujoco.mj_resetDataKeyframe(model, data, model.key("home").id)
data.ctrl[:] = model.key("home").ctrl
hip = ACTUATORS.index("front_left_second_joint")
frames, q, tau, t = [], [], [], []
for k in range(int(3.0 / model.opt.timestep)):
    if k == int(1.0 / model.opt.timestep):
        data.ctrl[hip] += 0.2
    mujoco.mj_step(model, data)
    q.append(data.qpos[model.jnt_qposadr[model.actuator_trnid[hip, 0]]]); tau.append(data.actuator_force[hip]); t.append(data.time)
    if k % 10 == 0:
        frames.append(frame(model, data))
show(frames)
fig, ax = plt.subplots(1, 2, figsize=(9, 2.4))
ax[0].plot(t, q); ax[0].set_ylabel("hip [rad]"); ax[1].plot(t, tau); ax[1].set_ylabel("torque [N·m]")
for a in ax: a.set_xlabel("t [s]"); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Step 2 — The environment

`WojtekJoystick` (`training/wojtek_rl/env.py`), built from the preset's `task.env` block.

- Actor obs (the robot's sensors): `joint_pos, joint_vel, last_act, command`. No IMU.
- Critic obs: privileged (true velocity, contacts, IMU). Training only.
- Action: 12 targets, `clip(anchor(height) + action * action_scale, low, high)`.
- Command: `[vx, vy, wz, height]`, resampled every 5 s.
- Reward: `dt * Σ raw * scale`, scales printed below.
- Randomized: mass, CoM, friction, gains, motor strength, latency, encoder offsets, noise; a push every 4 s; fall ends the episode.

View: 6 s of the env under zero action (hold the stance). The push at 4 s is the env's.

In [ ]:
from hydra import compose, initialize_config_dir
from mujoco import mjx
from omegaconf import OmegaConf
from wojtek_rl.registry import make_env

OVERRIDES = [f"+experiment={PRESET}", f"run_name={RUN_NAME}", f"seed={SEED}",
             f"++ppo.num_timesteps={TRAIN_STEPS}", f"++ppo.num_envs={NUM_ENVS}", "wandb.enable=false"]
with initialize_config_dir(config_dir=str(TRAINING / "wojtek_rl" / "conf"), version_base=None):
    hcfg = compose(config_name="config", overrides=OVERRIDES)

env_overrides = OmegaConf.to_container(hcfg.task.env, resolve=True)
env_overrides["sim"] = {"backend": "jax", "num_envs": 1}
env = make_env(hcfg.task.name, env_overrides)
cfg = env._config
active = dict(sorted({k: v for k, v in cfg.reward.scales.items() if v}.items(), key=lambda kv: -abs(kv[1])))
print("actor  :", env.actor_obs_names)
print("critic :", list(cfg.obs.privileged))
print(f"action {env.action_size} scale {tuple(cfg.action_scale)} | servo kp={cfg.pd_kp} kd={cfg.pd_kd} cap={cfg.max_torque} | episode {cfg.episode_length}")
print(f"command vx {tuple(cfg.command.vx)} vy {tuple(cfg.command.vy)} wz {tuple(cfg.command.wz)} height {tuple(cfg.command.height)}")
print("reward scales:", active)

In [ ]:
import jax.numpy as jp
reset, step = jax.jit(env.reset), jax.jit(env.step)
state = reset(jax.random.PRNGKey(0))
mjm = env.mj_model
frames, rewards = [], []
for i in range(300):
    state = step(state, jp.zeros(env.action_size))
    rewards.append(float(state.reward))
    if i % 2 == 0:
        frames.append(frame(mjm, mjx.get_data(mjm, state.data)))
print({k: v.shape for k, v in state.obs.items()}, "| command", np.round(np.asarray(state.info["command"]), 2))
for k, w in active.items():
    raw = float(state.metrics[f"reward/{k}"])
    print(f"  {k:18s} raw {raw:9.4f} x {w:8.4g} = {raw * w:8.4f}")
show(frames)
plt.figure(figsize=(9, 2)); plt.plot(np.arange(300) * cfg.ctrl_dt, rewards); plt.xlabel("t [s]"); plt.ylabel("reward/step"); plt.grid(alpha=0.3); plt.show()

## Step 3 — Config and PPO

Hydra: `conf/config.yaml` → task → `+experiment=<preset>` → overrides. Trainer: Brax PPO on MJX (MJWarp on CUDA), from the Playground Go1 config. A run writes `training/runs/<name>/run.json` and a checkpoint per eval.

In [ ]:
from wojtek_rl.train import build_ppo_params
ppo = build_ppo_params({}, smoke=False)
print({k: ppo[k] for k in ("num_envs", "batch_size", "num_minibatches", "unroll_length", "learning_rate", "entropy_cost", "discounting", "num_evals")})
print("network", tuple(ppo.network_factory.policy_hidden_layer_sizes), "| this run:", dict(hcfg.ppo))
print("dr:", {k: v.enable for k, v in hcfg.dr.items() if hasattr(v, "enable")})
print("terminal form: ./training/run.sh train " + " ".join(OVERRIDES))

## Step 4 — Train (GPU)

30 M steps ≈ minutes on a Colab GPU: enough to stand and track slowly, not to walk well. Raise `TRAIN_STEPS` for more. One change per run, into `OVERRIDES`: `++task.env.reward.scales.action_rate=-0.5`, `'++task.env.command.vx=[-0.4,0.6]'`, `++task.env.pd_kp=60 ++task.env.pd_kd=1.96`. If MJWarp rejects the GPU: `+task.env.sim.backend=jax`.

In [ ]:
has_gpu = any(d.platform == "gpu" for d in jax.devices())
if has_gpu and not (RUN_DIR / "run.json").exists():
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    with open(RUN_DIR / "train.log", "w") as log:
        proc = subprocess.Popen([sys.executable, "-m", "wojtek_rl.train", *OVERRIDES], cwd=TRAINING,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            if line.startswith(("steps", "done")) or "error" in line.lower():
                print(line, end="")
        proc.wait()
    print("exit", proc.returncode)
elif not has_gpu:
    print("no GPU: switch the runtime to GPU for this step")
else:
    print("run exists:", RUN_DIR)

In [ ]:
if (RUN_DIR / "run.json").exists():
    run = json.loads((RUN_DIR / "run.json").read_text())
    rows = re.findall(r"steps\s+([\d,]+)\s+reward\s+([-\d.]+)\s+ep_len\s+([\d.]+)", (RUN_DIR / "train.log").read_text())
    c = np.array([(int(s.replace(",", "")), float(r), float(l)) for s, r, l in rows]).reshape(-1, 3)
    print(run["status"], f"final reward {run['final_reward']}, kp={run['kp']} kd={run['kd']}")
    fig, ax = plt.subplots(1, 2, figsize=(9, 2.4))
    ax[0].plot(c[:, 0] / 1e6, c[:, 1], "o-"); ax[0].set_ylabel("eval reward")
    ax[1].plot(c[:, 0] / 1e6, c[:, 2], "o-"); ax[1].set_ylabel("episode length")   # short episodes = falling
    for a in ax: a.set_xlabel("M steps"); a.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("no run")

## Step 5 — Watch the checkpoint

`wojtek_rl.eval` rebuilds the env from `run.json`, loads the last checkpoint, runs stand → trot → turn → trot → stand, and renders it with the command bar, torque strip and joint traces.

In [ ]:
if (RUN_DIR / "run.json").exists():
    clip = OUT / f"{RUN_NAME}_demo.mp4"
    run_module("wojtek_rl.eval", "--run", f"runs/{RUN_NAME}", "--scenario", "demo_sequence", "--out", str(clip))
    display(Video(str(clip), embed=True, width=640))
    # Numbers: run_module("wojtek_rl.report", "--run", f"runs/{RUN_NAME}", cpu=True)
else:
    print("no run")

## Step 6 — Export for the robot

`policy.npz` (normalizer + MLP) + `policy_meta.json` (schema-2 contract: obs layout, anchor, scale, bounds, command box, servo gains, `tau_ff`). The robot runs this pair through `WojtekPolicy` (NumPy) at 50 Hz. Policies observing the gait clock are refused.

In [ ]:
def describe(meta, title):
    pd, tff = meta.get("pd", {}), meta.get("tau_ff") or {}
    print(f"{title}: {meta['run_name']} | obs {meta['obs_layout']} | action {meta['action_size']} | "
          f"servo kp={pd.get('kp')} kd={pd.get('kd')} cap={pd.get('max_torque')} | tau_ff={'±%g' % tff['scale'] if tff.get('enable') else 'off'}")

if (RUN_DIR / "run.json").exists():
    if not (EXPORT_DIR / "policy_meta.json").exists():
        run_module("wojtek_rl.export_policy", "--run", f"runs/{RUN_NAME}", cpu=True)
    w = np.load(EXPORT_DIR / "policy.npz")
    print({k: w[k].shape for k in w.files})
    describe(json.loads((EXPORT_DIR / "policy_meta.json").read_text()), "mine")
else:
    print("no run")

## Step 7 — Pick a keeper

Keepers: Hugging Face repos `<HF_ORGANIZATION>/<name>` with checkpoint, `run.json`, battery, videos and the exported pair. A reference resolves through `wojtek_policy.policy_source` (the robot's resolver) into `ros/policies/`.

| repo | servo | note |
|---|---|---|
| `wojtek-quiet-locomotion` | kp40/kd0.8/12 + tau_ff | pinned default on the robot |
| `wojtek-stiff-height-locomotion` | kp40/kd1.6/9 | live height command |
| `wojtek-stiff-locomotion`, `-v2` | kp40/kd1.6/9 | `locomotion_stiff_v1` recipe |
| `wojtek-stiff-kp80-locomotion`, `-kp90-` | kp80 / kp90 | stiffness ladder |
| `wojtek-springy-locomotion`, `-v2` | kp20/kd1 | clock-free baseline |
| `wojtek-terrain-blind-locomotion-v42` | kp40/kd1.6 | IMU in actor, terrain |

In [ ]:
sys.path.insert(0, str(paths.WOJTEK_POLICY_PKG))
if paths.HF_ORGANIZATION:
    os.environ.setdefault("HF_ORGANIZATION", paths.HF_ORGANIZATION)
from wojtek_policy.policy_source import default_policy, resolve_policy
from wojtek_rl.np_policy import load_policy_runtime

ORG = os.environ.get("HF_ORGANIZATION", "")
KEEPERS = ["wojtek-quiet-locomotion", "wojtek-stiff-height-locomotion", "wojtek-stiff-locomotion", "wojtek-stiff-locomotion-v2",
           "wojtek-stiff-kp80-locomotion", "wojtek-stiff-kp90-locomotion", "wojtek-springy-locomotion",
           "wojtek-springy-locomotion-v2", "wojtek-terrain-blind-locomotion-v42"]
if ORG:
    try:
        from huggingface_hub import HfApi
        KEEPERS = sorted(m.id.split("/", 1)[1] for m in HfApi().list_models(author=ORG) if m.id.split("/", 1)[1].startswith("wojtek-")) or KEEPERS
    except Exception as e:
        print("HF listing failed:", type(e).__name__)
print("robot pin:", default_policy() or "(HF_ORGANIZATION unset)")

import ipywidgets as widgets
keeper_widget = widgets.Dropdown(options=KEEPERS, value=KEEPER_NAME if KEEPER_NAME in KEEPERS else KEEPERS[0], description="keeper")
display(keeper_widget)

In [ ]:
CANDIDATES = {}
pin, name = default_policy(), keeper_widget.value
ref = pin if pin and pin.split("/")[1].split("@")[0] == name else (f"{ORG}/{name}" if ORG else "")
if ref:
    try:
        CANDIDATES["keeper"] = load_policy_runtime(ref)
        print("keeper <-", resolve_policy(ref).source)
    except Exception as e:
        print("keeper unavailable:", e)
if (EXPORT_DIR / "policy_meta.json").exists():
    CANDIDATES["mine"] = load_policy_runtime(EXPORT_DIR)
    print("mine   <-", EXPORT_DIR)
for n, p in CANDIDATES.items():
    describe(p.meta, n)

## Step 8 — Head-to-head in MuJoCo

The robot's control loop with MuJoCo as the hardware: encoders (+ IMU if the layout has it) → `policy.step` → targets and `tau_ff` → 5 physics substeps. Each policy runs on the servo gains from its own contract. No randomization or noise; a push is optional. `hold_home` = do nothing.

Scenario: stand 3 s, trot 0.5 m/s, turn 0.7 rad/s, trot, stand.

In [ ]:
from wojtek_rl.np_policy import actuator_addresses, gravity_from_quat

FALL_HEIGHT, FALL_GZ = 0.06, -0.4


class HoldHome:
    def __init__(self, m):
        self.meta = {"pd": {"kp": float(m.actuator_gainprm[0, 0]), "kd": float(-m.actuator_biasprm[0, 2]), "max_torque": float(m.actuator_forcerange[0, 1])}}
        self.ctrl_dt, self.tau_ff_enabled = 0.02, False
        self._home = m.key("home").ctrl.copy()
    def reset(self): pass
    def step(self, gyro, gravity, q, dq, command): return self._home


def model_for(policy):
    m = mujoco.MjModel.from_xml_path(str(paths.SCENE_XML))
    pd = policy.meta.get("pd")
    if pd:
        m.actuator_gainprm[:, 0] = pd["kp"]; m.actuator_biasprm[:, 1] = -pd["kp"]; m.actuator_biasprm[:, 2] = -pd["kd"]
        m.actuator_forcerange[:, 0] = -pd["max_torque"]; m.actuator_forcerange[:, 1] = pd["max_torque"]
    return m


def rollout(policy, command_at, n_steps, *, seed=0, push_at=None, push_vel=0.6, render_every=2):
    m = model_for(policy); d = mujoco.MjData(m)
    mujoco.mj_resetDataKeyframe(m, d, m.key("home").id); mujoco.mj_forward(m, d)
    qadr, vadr = actuator_addresses(m)
    substeps = round(policy.ctrl_dt / m.opt.timestep)
    tau_ff_on = bool(getattr(policy, "tau_ff_enabled", False))
    gyro_adr = m.sensor("angular-velocity").adr[0]
    rng = np.random.default_rng(seed)
    policy.reset()
    log = {k: [] for k in ("t", "cmd", "v_body", "wz", "height", "torque", "dq")}
    frames, fell_at, qinv, v_body = [], None, np.zeros(4), np.zeros(3)
    for i in range(n_steps):
        cmd = np.asarray(command_at(i), np.float32)
        targets = policy.step(d.sensordata[gyro_adr:gyro_adr + 3].copy(), gravity_from_quat(*d.qpos[3:7]), d.qpos[qadr].copy(), d.qvel[vadr].copy(), cmd)
        d.ctrl[:] = targets
        if tau_ff_on:
            d.qfrc_applied[:] = 0.0; d.qfrc_applied[vadr] = policy.last_tau_ff
        if push_at is not None and i == push_at:
            v = rng.uniform(-1, 1, 2); d.qvel[:2] += v / (np.linalg.norm(v) + 1e-6) * push_vel
        for _ in range(substeps):
            mujoco.mj_step(m, d)
        mujoco.mju_negQuat(qinv, d.qpos[3:7]); mujoco.mju_rotVecQuat(v_body, d.qvel[:3], qinv)
        log["t"].append(i * policy.ctrl_dt); log["cmd"].append(cmd[:3]); log["v_body"].append(v_body.copy())
        log["wz"].append(d.sensordata[gyro_adr + 2]); log["height"].append(d.qpos[2])
        log["torque"].append(d.actuator_force.copy()); log["dq"].append(d.qvel[vadr].copy())
        if i % render_every == 0:
            frames.append(frame(m, d))
        if d.qpos[2] < FALL_HEIGHT or gravity_from_quat(*d.qpos[3:7])[2] > FALL_GZ:
            fell_at = i; break
    return {k: np.asarray(v) for k, v in log.items()}, fell_at, frames


def summarize(log, fell_at, dt=0.02):
    cmd, v, wz = log["cmd"], log["v_body"], log["wz"]
    moving = np.abs(cmd).max(axis=1) > 0.05
    err = np.hypot(v[:, 0] - cmd[:, 0], v[:, 1] - cmd[:, 1])
    dq = log["dq"] - log["dq"].mean(axis=0, keepdims=True)
    power = np.abs(np.fft.rfft(dq, axis=0)) ** 2; freqs = np.fft.rfftfreq(dq.shape[0], d=dt)
    return {"fell": "no" if fell_at is None else f"{fell_at * dt:.1f} s",
            "vel_err_rms": float(np.sqrt(np.mean(err[moving] ** 2))) if moving.any() else float("nan"),
            "wz_err_rms": float(np.sqrt(np.mean((wz[moving] - cmd[moving, 2]) ** 2))) if moving.any() else float("nan"),
            "height": float(log["height"].mean()),
            "torque_p90": float(np.percentile(np.abs(log["torque"]), 90)),
            "vibration": float(power[freqs > 5.0].sum() / max(power[freqs > 0].sum(), 1e-12))}


def demo_sequence(i):
    if i < 150: return (0.0, 0.0, 0.0)
    if i < 450: return (0.5, 0.0, 0.0)
    if i < 750: return (0.0, 0.0, 0.7)
    if i < 1050: return (0.5, 0.0, 0.0)
    return (0.0, 0.0, 0.0)

PUSH_AT = None      # e.g. 300 = a shove mid-trot
contenders = {"hold_home": HoldHome(model), **CANDIDATES}
runs = {n: rollout(p, demo_sequence, 1200, push_at=PUSH_AT) for n, p in contenders.items()}

names = list(runs)
metrics = {n: summarize(*runs[n][:2]) for n in names}
print(f"{'':12s}" + "".join(f"{n:>12s}" for n in names))
for k in next(iter(metrics.values())):
    print(f"{k:12s}" + "".join(f"{metrics[n][k]:>12.3f}" if isinstance(metrics[n][k], float) else f"{metrics[n][k]:>12s}" for n in names))

length = max(len(r[2]) for r in runs.values())
show([np.hstack([r[2][min(k, len(r[2]) - 1)] for r in runs.values()]) for k in range(length)])   # left to right: names
fig, ax = plt.subplots(3, 1, figsize=(9, 5.5), sharex=True)
log0 = runs[names[-1]][0]
ax[0].plot(log0["t"], log0["cmd"][:, 0], "k--"); ax[1].plot(log0["t"], log0["cmd"][:, 2], "k--")
for n in names:
    log = runs[n][0]
    ax[0].plot(log["t"], log["v_body"][:, 0], label=n); ax[1].plot(log["t"], log["wz"]); ax[2].plot(log["t"], log["height"])
ax[0].set_ylabel("vx [m/s]"); ax[1].set_ylabel("wz [rad/s]"); ax[2].set_ylabel("height [m]"); ax[2].set_xlabel("t [s]"); ax[0].legend(fontsize=8)
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Step 9 — Your command

`command_at(i) -> (vx, vy, wz)` or `(vx, vy, wz, height)`, 50 steps per second. Edit and rerun.

In [ ]:
CUSTOM = lambda i: (0.0, 0.0, -0.8)     # spin right
N = 300
custom = {n: rollout(p, CUSTOM, N, push_at=None) for n, p in contenders.items()}
for n, (log, fell_at, _) in custom.items():
    print(f"{n:10s} vx {log['v_body'][50:, 0].mean():+.2f}  wz {log['wz'][50:].mean():+.2f}  fell={fell_at}  vibration {summarize(log, fell_at)['vibration']:.2f}")
length = max(len(r[2]) for r in custom.values())
show([np.hstack([r[2][min(k, len(r[2]) - 1)] for r in custom.values()]) for k in range(length)])

## Next

- Judge runs with `wojtek_rl.report` and `wojtek_rl.courses`, not reward. [Training lessons](../skills/brax-locomotion-training/references/wojtek-training-lessons.md); [configuration reference](../training/docs/configuration.md).
- Full runs: unique `run_name`, WandB on, `training/jobs/` payloads on a cluster.
- Keeper: upload the exported pair next to the checkpoint on HF. Deploy: `./ros/deploy.sh --policy <org/name@commit>` + manual arming ([ros/README.md](../ros/README.md)). Human-authorized, never from a notebook.